# Seasonal Trends in ARCHER2 Node-Hour Usage

This notebook looks at how workload changes across seasons for the busiest software on ARCHER2.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

data_path = Path("../data/archer2_HPC_data_without_overall_and_aOut.csv")
hpc_data = pd.read_csv(data_path)

# Clean the data
hpc_data["Year"] = pd.to_numeric(hpc_data["Year"], errors="coerce")
hpc_data["Month"] = pd.to_numeric(hpc_data["Month"], errors="coerce")
hpc_data["Nodeh"] = pd.to_numeric(hpc_data["Nodeh"], errors="coerce").fillna(0)

season_order = ["Winter", "Spring", "Summer", "Autumn"]
hpc_data["Season"] = pd.Categorical(
    hpc_data["Season"],
    categories=season_order,
    ordered=True
)

print("Dataset loaded:", hpc_data.shape)
print(hpc_data.head())

In [ ]:
# Find the top 5 software by total node-hours
overall_top5 = (
    hpc_data.groupby("Software")["Nodeh"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

print("Top 5 software overall:")
print(overall_top5)

top_names = overall_top5.index.tolist()
filtered = hpc_data[hpc_data["Software"].isin(top_names)]

In [ ]:
# Aggregate seasonal totals for the selected software
seasonal_trend = (
    filtered.groupby(["Season", "Software"], observed=True)["Nodeh"]
    .sum()
    .reset_index()
)

pivot_df = seasonal_trend.pivot_table(
    index="Season",
    columns="Software",
    values="Nodeh",
    aggfunc="sum",
    observed=True
).fillna(0)

pivot_df = pivot_df.reindex(season_order)
print(pivot_df)

In [ ]:
# Plot the seasonal trend across all years
pivot_df.plot(
    kind="line",
    figsize=(12, 8),
    linewidth=2,
    marker="*",
    cmap="tab10",
    markersize=7
)

plt.title("Seasonal Node-Hour Trend for the Top 5 Software (2021–2026)")
plt.xlabel("Season")
plt.ylabel("Node-hours")
plt.grid(axis="y", linestyle="-", alpha=0.6)
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()